In [ ]:
import pandas as pd

FILE_PATH = "../../data/processed/MASTER_DATASET_LABELED.csv"
df = pd.read_csv(FILE_PATH, low_memory=False)

counts = df['domain_label'].value_counts()
percentages = df['domain_label'].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': percentages.round(2)
})

print("Final Classification Statistics:")
print(summary)

print(f"\nTotal rows processed: {len(df)}")

In [ ]:
OUTPUT_FILE = "../../data/processed/TARGET_VACANCIES.csv"

target_domains = ['IT', 'Finance', 'HR']
filtered_df = df[df['domain_label'].isin(target_domains)].copy()

print(f"Total target rows (IT, Finance, HR): {len(filtered_df)}")

filtered_df['parsed_date'] = pd.to_datetime(filtered_df['date_posted'], errors='coerce', utc=True)

clean_ts = filtered_df['timestamp'].astype(str).str.replace(r'\.0$', '', regex=True)
filtered_df['parsed_ts'] = pd.to_datetime(clean_ts, format='%Y%m%d%H%M%S', errors='coerce', utc=True)

filtered_df['final_date'] = filtered_df['parsed_date'].fillna(filtered_df['parsed_ts'])

initial_count = len(filtered_df)
filtered_df = filtered_df.dropna(subset=['final_date'])
dropped_dates = initial_count - len(filtered_df)
print(f"Dropped {dropped_dates} rows due to completely unreadable/missing dates.")

filtered_df['year_month'] = filtered_df['final_date'].dt.tz_localize(None).dt.to_period('M')

filtered_df = filtered_df.drop(columns=['parsed_date', 'parsed_ts'])

filtered_df.to_csv(OUTPUT_FILE, index=False)
print(f"\n usable rows for Phase 3: {len(filtered_df)}")
print(f"Dataset saved as: {OUTPUT_FILE}")

In [ ]:
import pandas as pd
import os
from IPython.display import display

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
FILE_PATH = os.path.join(BASE_DIR, "data", "processed", "TARGET_VACANCIES.csv")

df = pd.read_csv(FILE_PATH, usecols=['year_month', 'domain_label'], low_memory=False)
df = df.dropna(subset=['year_month', 'domain_label'])

df['year'] = df['year_month'].astype(str).str[:4].astype(int)

counts = pd.crosstab(df['year'], df['domain_label'])
counts['TOTAL'] = counts.sum(axis=1)

pct = pd.crosstab(df['year'], df['domain_label'], normalize='index') * 100
pct = pct.round(1).astype(str) + "%"

print("\033[1mABSOLUTE COUNTS PER YEAR\033[0m")
display(counts)

print("\n\033[1mPERCENTAGE STRUCTURE PER YEAR\033[0m")
display(pct)

In [ ]:
import pandas as pd
import os
from IPython.display import display

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
TARGET_COUNTRIES = ['CH', 'PL', 'DE', 'US']
STOP_WORDS = [
    'Finance', 'Financial Services', 'Banking', 'Disabilities', 
    'Diversity And Inclusion', 'Job Descriptions', 'Mental Health', 
    'Net Zero', 'Economic Growth', 'Retirement Planning', 'Economy',
    'Personal Finance'
]

def build_evolution_table(file_name, title, country=None, top_n=15):
    path = os.path.join(BASE_DIR, "data", "processed", file_name)
    df = pd.read_csv(path)
    
    skill_col = 'hard_skills' if 'HARD' in file_name else 'soft_skills'
    
    if country:
        df = df[df['country'] == country].copy()
        
    df = df[~df[skill_col].isin(STOP_WORDS)].copy()
    
    df['rank'] = df.groupby('year')['percentage'].rank(method='first', ascending=False).astype(int)
    df_top = df[df['rank'] <= top_n].copy()
    
    df_top['display_val'] = df_top[skill_col] + " (" + df_top['percentage'].round(1).astype(str) + "%)"
    pivot_table = df_top.pivot(index='rank', columns='year', values='display_val')
    
    print(f"\n\033[1m{title}\033[0m")
    display(pivot_table.fillna("-"))

build_evolution_table("FINANCE_TOP_30_GLOBAL_HARD.csv", "GLOBAL: HARD SKILLS EVOLUTION")
build_evolution_table("FINANCE_TOP_30_GLOBAL_SOFT.csv", "GLOBAL: SOFT SKILLS EVOLUTION")

for c in TARGET_COUNTRIES:
    build_evolution_table("FINANCE_TOP_30_BY_COUNTRY_HARD.csv", f"{c}: HARD SKILLS EVOLUTION", country=c)
    build_evolution_table("FINANCE_TOP_30_BY_COUNTRY_SOFT.csv", f"{c}: SOFT SKILLS EVOLUTION", country=c)

In [ ]:
import pandas as pd
import os
from IPython.display import display

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
TARGET_COUNTRIES = ['CH', 'PL', 'DE', 'US']

STOP_WORDS = [
    'Information Technology', 'Computer Science', 'Disabilities', 
    'Diversity And Inclusion', 'Job Descriptions', 'Mental Health', 
    'Net Zero', 'Economic Growth', 'Retirement Planning', 'Economy',
    'Personal Finance', 'Legislation', 'Office Equipment', 'Motorcycles', 
    'LinkedIn', 'Yoga', 'Finance', 'Banking', 'Supply Chain', 
    'Requisition', 'Insurance Policies', 'Regulatory Requirements', 
    'Accounting', 'Risk Appetite', 'Regulatory Compliance', 'Telecommuting',
    'Financial Services', 'Investment Banking', 'Asset Management', 
    'Commercial Banking', 'Investment Management', 'Business Requirements'
]

def build_evolution_table(file_name, title, country=None, top_n=15):
    path = os.path.join(BASE_DIR, "data", "processed", file_name)
    df = pd.read_csv(path)
    
    skill_col = 'hard_skills' if 'HARD' in file_name else 'soft_skills'
    
    if country:
        df = df[df['country'] == country].copy()
        
    df = df[~df[skill_col].isin(STOP_WORDS)].copy()
    
    df['rank'] = df.groupby('year')['percentage'].rank(method='first', ascending=False).astype(int)
    df_top = df[df['rank'] <= top_n].copy()
    
    df_top['display_val'] = df_top[skill_col] + " (" + df_top['percentage'].round(1).astype(str) + "%)"
    pivot_table = df_top.pivot(index='rank', columns='year', values='display_val')
    
    print(f"\n\033[1m{title}\033[0m")
    display(pivot_table.fillna("-"))

build_evolution_table("IT_TOP_30_GLOBAL_HARD.csv", "GLOBAL: IT HARD SKILLS EVOLUTION")
build_evolution_table("IT_TOP_30_GLOBAL_SOFT.csv", "GLOBAL: IT SOFT SKILLS EVOLUTION")

for c in TARGET_COUNTRIES:
    build_evolution_table("IT_TOP_30_BY_COUNTRY_HARD.csv", f"{c}: IT HARD SKILLS EVOLUTION", country=c)
    build_evolution_table("IT_TOP_30_BY_COUNTRY_SOFT.csv", f"{c}: IT SOFT SKILLS EVOLUTION", country=c)

In [ ]:
import pandas as pd
import os
from IPython.display import display

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
TARGET_COUNTRIES = ['CH', 'PL', 'DE', 'US']

STOP_WORDS = [
    'Human Resources', 'Job Descriptions', 'Disabilities', 
    'Net Zero', 'Economic Growth', 'Economy', 'Labor Law'
]

def build_evolution_table(file_name, title, country=None, top_n=15):
    path = os.path.join(BASE_DIR, "data", "processed", file_name)
    if not os.path.exists(path):
        return
        
    df = pd.read_csv(path)
    skill_col = 'hard_skills' if 'HARD' in file_name else 'soft_skills'
    
    if country:
        df = df[df['country'] == country].copy()
        
    df = df[~df[skill_col].isin(STOP_WORDS)].copy()
    
    if df.empty:
        return
        
    df['rank'] = df.groupby('year')['percentage'].rank(method='first', ascending=False).astype(int)
    df_top = df[df['rank'] <= top_n].copy()
    
    df_top['display_val'] = df_top[skill_col] + " (" + df_top['percentage'].round(1).astype(str) + "%)"
    pivot_table = df_top.pivot(index='rank', columns='year', values='display_val')
    
    print(f"\n\033[1m{title}\033[0m")
    display(pivot_table.fillna("-"))

build_evolution_table("HR_TOP_30_GLOBAL_HARD.csv", "GLOBAL: HR HARD SKILLS EVOLUTION")
build_evolution_table("HR_TOP_30_GLOBAL_SOFT.csv", "GLOBAL: HR SOFT SKILLS EVOLUTION")

for c in TARGET_COUNTRIES:
    build_evolution_table("HR_TOP_30_BY_COUNTRY_HARD.csv", f"{c}: HR HARD SKILLS EVOLUTION", country=c)
    build_evolution_table("HR_TOP_30_BY_COUNTRY_SOFT.csv", f"{c}: HR SOFT SKILLS EVOLUTION", country=c)